# Step 3. Find shifts. Currently they are adjusted from the ones Peter got.

In [ ]:
import numpy as np
import cupy as cp
import h5py
import matplotlib.pyplot as plt
import scipy
from holotomocupy.utils import *

## Init

In [ ]:
ntheta = 360
ids = np.arange(0, 1800, 1800 / ntheta).astype('int')


path_out = '/data2/vnikitin/atomium_rec/20240924/AtomiumS2/'
file_out = f'data.h5'

with h5py.File(f'{path_out}/data.h5') as fid:
    detector_pixelsize = fid['/exchange/detector_pixelsize'][0]    
    focustodetectordistance = fid['/exchange/focusdetectordistance'][0]    
    z1 = fid['/exchange/z1'][:] 
    theta = -fid['/exchange/theta'][ids, 0] / 180 * np.pi
    energy = fid['/exchange/energy'][0] 
    shifts = fid['/exchange/shifts'][ids]    
    shape = np.array(fid[f'/exchange/data0'].shape)

wavelength = 1.24e-09 / energy
z2 = focustodetectordistance - z1
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]
distances = (z1 * z2) / focustodetectordistance * norm_magnifications**2
voxelsize = detector_pixelsize / magnifications[0]

n = shape[1]
ndist=len(z1)
print(f'{energy=}')
print(f'{z1=}')
print(f'{focustodetectordistance=}')
print(f'{detector_pixelsize=}')
print(f'{magnifications=}')
print(f'{voxelsize=}')
print(f'{distances=}')

### Motion shifts, alignment for a reference plane. Initially given with random shifts include, we subtract random shifts.

In [ ]:
import math
from holotomocupy.shift import Shift
with h5py.File(f'/data2/vnikitin/atomium/20240924/AtomiumS2/AtomiumS2_HT_007nm_/AtomiumS2_HT_007nm_rec_.nx','r') as fid:
    print(fid['/entry/data/data'].shape)
    d = fid['/entry/data/data'][ids]        

norm_magnifications = np.array([1],dtype='float32')
nz,n = d.shape[1:]
cl_shift = Shift(n, n,nz,nz, np.array([1],dtype='float32'),'float32')

correct3d_shifts = np.loadtxt(f'/data2/vnikitin/brain_rec/20251115/Y350a/correct_correct3D_1234.txt')[:4500][:, ::-1].astype('float32')
correct3d_shifts = np.tile(correct3d_shifts[:, np.newaxis], (1, ndist, 1))

ids = np.arange(0, 4500, 4500 / ntheta).astype('int')
correct3d_shifts = correct3d_shifts[ids]
correct3d_shifts*=20/7
# correct3d_shifts[...,1] = 0
pos = -correct3d_shifts[:,0]
pos[...,1] *= 0
pos[...,1] += -(-26.027329999999893)
pos[...,0] -= 400

plt.plot(pos[...,1],'.')
plt.plot(pos[...,0],'.')
plt.show()
nchunk = 2
sd = np.empty_like(d)
for k in range(0,math.ceil(len(d)/nchunk)):    
    st = k*nchunk
    end = min((k+1)*nchunk,len(d))
    dd = cp.array(d[st:end])
    ppos = cp.array(pos[st:end])
    
    sd[st:end] = cl_shift.curlyS(dd,ppos,0).get()    

In [ ]:
mshow(-sd[0],True,vmax=0.6,vmin=-0.1)

In [ ]:
# def center_of_mass(img, mask=None, bg=None, clamp_negative=True):
#     """
#     Center of mass (COM) of a 2D image using nonnegative weights.

#     Parameters
#     ----------
#     img : (H, W) array
#         Image where "mass" is represented by intensity (often -log transmission).
#     mask : (H, W) bool/0-1 array, optional
#         Region to include in the COM calculation.
#     bg : float, optional
#         Background level to subtract before computing COM. If None, no subtraction.
#     clamp_negative : bool
#         If True, set negative weights to 0 after background subtraction.

#     Returns
#     -------
#     (x_com, y_com) : floats
#         COM in pixel coordinates (x = column index, y = row index).
#     """
#     img = cp.asarray(img, dtype=np.float64)
#     H, W = img.shape

#     w = img.copy()
#     if bg is not None:
#         w = w - float(bg)

#     if clamp_negative:
#         w[w < 0] = 0.0

#     if mask is not None:
#         w *= cp.asarray(mask, dtype=np.float64)

#     s = w.sum()
#     if s <= 0:
#         # Fallback: geometric center
#         return (W - 1) / 2.0, (H - 1) / 2.0

#     yy, xx = cp.indices((H, W), dtype=np.float64)
#     x_com = (xx * w).sum() / s
#     y_com = (yy * w).sum() / s
#     return x_com, y_com


# def border_median_background(img, border=16):
#     """Median background estimated from a border frame of width `border`."""
#     img = np.asarray(img)
#     H, W = img.shape
#     b = int(max(1, min(border, H//2, W//2)))
#     top = img[:b, :].ravel()
#     bot = img[-b:, :].ravel()
#     lef = img[:, :b].ravel()
#     rig = img[:, -b:].ravel()
#     return np.median(np.concatenate([top, bot, lef, rig]))

In [ ]:
# import cv2
# def _find_min_max(data):
#     """Find min and max values according to histogram"""

#     mmin = np.zeros(data.shape[0], dtype='float32')
#     mmax = np.zeros(data.shape[0], dtype='float32')

#     for k in range(data.shape[0]):
#         h, e = np.histogram(data[k][:], 1000)
#         stend = np.where(h > np.max(h)*0.005)
#         st = stend[0][0]
#         end = stend[0][-1]
#         mmin[k] = e[st]
#         mmax[k] = e[end+1]

#     return mmin, mmax

# def register_shift_sift(datap1, datap2, th=0.5):
#     """Find shifts via SIFT detecting features"""

#     mmin, mmax = _find_min_max(datap1)
#     sift = cv2.SIFT_create()
#     shifts = np.zeros([datap1.shape[0], 2], dtype='float32')
#     for id in range(datap1.shape[0]):
#         tmp1 = ((datap2[id]-mmin[id]) /
#                 (mmax[id]-mmin[id])*255)
#         tmp1[tmp1 > 255] = 255
#         tmp1[tmp1 < 0] = 0
#         tmp2 = ((datap1[id]-mmin[id]) /
#                 (mmax[id]-mmin[id])*255)
#         tmp2[tmp2 > 255] = 255
#         tmp2[tmp2 < 0] = 0
#         # find key points
#         tmp1 = tmp1.astype('uint8')
#         tmp2 = tmp2.astype('uint8')

#         kp1, des1 = sift.detectAndCompute(tmp1, None)
#         kp2, des2 = sift.detectAndCompute(tmp2, None)
#         # cv2.imwrite('/data/Fister_rec/original_image_right_keypoints.png',cv2.drawKeypoints(tmp1,kp1,None))
#         # cv2.imwrite('/data/Fister_rec/original_image_left_keypoints.png',cv2.drawKeypoints(tmp2,kp2,None))
#         match = cv2.BFMatcher()
#         matches = match.knnMatch(des1, des2, k=2)
#         good = []
#         for m, n in matches:
#             if m.distance < th*n.distance:
#                 good.append(m)
#         if len(good) == 0:
#             print('no features found')
#             exit()
#         draw_params = dict(matchColor=(0, 255, 0),
#                            singlePointColor=None,
#                            flags=2)
#         tmp3 = cv2.drawMatches(tmp1, kp1, tmp2, kp2,
#                                good, None, **draw_params)
#         cv2.imwrite("matches.jpg", tmp3)
#         src_pts = np.float32(
#             [kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
#         dst_pts = np.float32(
#             [kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
#         shift = (src_pts-dst_pts)[:, 0, :]
#         shifts[id] = np.mean(shift, axis=0)[::-1]
#     return shifts, len(good)


In [ ]:
# sx=np.zeros(ntheta)
# sy=np.zeros(ntheta)
# import dxchange
# # dxchange.write_tiff_stack(sd,'/data2/vnikitin/ttt/t',overwrite=True)
# for i,k in enumerate(range(ntheta)):
#     d0 = -cp.array(sd[k]).copy()
#     # d0[d0<0]=0
#     d0=(d0+0.2)*(cp.array(mask))
    
#     sx[i],sy[i]=center_of_mass(d0)
#     if k%100==0:    
#         print(sx[i],sy[i])
#         mshow(d0,True)

# plt.plot(sx)
# plt.show()
# plt.plot(sy)
# plt.show()
# # plt.plot(dx)
# # plt.plot(dy)
# # proj_aligned = apply_shifts(proj, dx, dy)
# # sss

In [ ]:
import math
from holotomocupy.shift import Shift
with h5py.File(f'/data2/vnikitin/atomium/20240924/AtomiumS2/AtomiumS2_HT_007nm_/AtomiumS2_HT_007nm_rec_.nx','r') as fid:
    d = fid['/entry/data/data'][0:ntheta]        

for it,t in enumerate(np.linspace(0.7,1.3,11)):
    norm_magnifications = np.array([1],dtype='float32')
    print(it,t)
    nz,n = d.shape[1:]
    cl_shift = Shift(n, n,nz,nz, np.array([1],dtype='float32'),'float32')

    correct3d_shifts = np.loadtxt(f'/data2/vnikitin/brain_rec/20251115/Y350a/correct_correct3D_1234.txt')[:4500][:, ::-1].astype('float32')
    correct3d_shifts = np.tile(correct3d_shifts[:, np.newaxis], (1, ndist, 1))

    ids = np.arange(0, 4500, 4500 / ntheta).astype('int')
    correct3d_shifts = correct3d_shifts[ids]
    correct3d_shifts*=20/7
    # correct3d_shifts[...,1] = 0
    pos = -correct3d_shifts[:,0]*t
    pos[...,1] *= 0
    pos[...,1] += -(-26.027329999999893)
    plt.plot(pos[...,1],'.')
    plt.plot(pos[...,0],'.')
    plt.show()
    nchunk = 32
    sd = np.empty_like(d)
    for k in range(0,math.ceil(len(d)/nchunk)):    
        st = k*nchunk
        end = min((k+1)*nchunk,len(d))
        dd = cp.array(d[st:end])
        ppos = cp.array(pos[st:end])
        
        sd[st:end] = cl_shift.curlyS(dd,ppos,0).get()    

    from types import SimpleNamespace
    from holotomocupy.tomo_batched import TomoBatched
    args = SimpleNamespace()
    args.ngpus = cp.cuda.runtime.getDeviceCount()

    args.ntheta = ntheta
    args.nobj = n
    args.nzobj = 2

    args.nchunk = 1
    args.show = True
    args.theta = theta
    args.mask_r = 1.2

    cl_rec = TomoBatched(args)

    import dxchange


    rec = cl_rec.rec_tomo(sd.astype('complex64'), 31).astype('float32')
    dxchange.write_tiff_stack(rec[rec.shape[0]//2],f'/data2/rz{it}',overwrite=True)
    # dxchange.write_tiff_stack(rec[:,rec.shape[1]//2],f'/data2/ry{it}',overwrite=True)


In [ ]:

# rec = cl_rec.rec_tomo(sd[:,sd.shape[1]//2:sd.shape[1]//2+1].astype('complex64'), 31).astype('float32')
# mshow(rec[0],True)
# dxchange.write_tiff_stack(rec,f'/data2/rz{it}',overwrite=True)
# # dxchange.write_tiff_stack(rec[:,rec.shape[1]//2],f'/data2/ry{it}',overwrite=True)